**Fresh BERT Fine-Tuning Experiment**

A fresh BERT-base model will be trained using unweighted Cross-Entropy Loss with an improved optimization strategy.

This experiment investigates whether better control of the learning process can improve generalization compared with the previous unweighted BERT baseline.

The experiment uses:

AdamW optimizer with learning rate 2e-5
Weight decay of 0.01
10% linear learning-rate warm-up
Linear learning-rate decay
Gradient clipping with maximum norm 1.0
Early stopping based on validation Macro-F1
Maximum of 8 epochs

Importing Required Libraries

In [ ]:
# Install required libraries
!pip install -q transformers datasets

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

print("PyTorch version:", torch.__version__)
print("Transformers version:", __import__("transformers").__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Transformers version: 5.16.1
Device: cuda
GPU: Tesla T4


**Load the Medical Abstract Dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("TimSchopf/medical_abstracts")

print(dataset)

README.md:   0%|          | 0.00/4.96k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.67MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.94MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11550 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2888 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['condition_label', 'medical_abstract'],
        num_rows: 11550
    })
    test: Dataset({
        features: ['condition_label', 'medical_abstract'],
        num_rows: 2888
    })
})


In [ ]:
print(dataset["train"])
print(dataset["test"])

print(dataset["train"].features)

Dataset({
    features: ['condition_label', 'medical_abstract'],
    num_rows: 11550
})
Dataset({
    features: ['condition_label', 'medical_abstract'],
    num_rows: 2888
})
{'condition_label': Value('int64'), 'medical_abstract': Value('string')}


In [ ]:
#Dataset->DataFrame
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

print("\nTraining columns:")
print(train_df.columns)

print("\nClass distribution:")
print(train_df["condition_label"].value_counts().sort_index())

Training samples: 11550
Test samples: 2888

Training columns:
Index(['condition_label', 'medical_abstract'], dtype='object')

Class distribution:
condition_label
1    2530
2    1195
3    1540
4    2441
5    3844
Name: count, dtype: int64


In [ ]:
#Validation Split
train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["condition_label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training samples  :", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples      :", len(test_df))

print("\nTraining distribution:")
print(train_df["condition_label"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["condition_label"].value_counts().sort_index())

Training samples  : 9817
Validation samples: 1733
Test samples      : 2888

Training distribution:
condition_label
1    2150
2    1016
3    1309
4    2075
5    3267
Name: count, dtype: int64

Validation distribution:
condition_label
1    380
2    179
3    231
4    366
5    577
Name: count, dtype: int64


In [ ]:
#Defining the model
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512
NUM_CLASSES = 5

print("Model:", MODEL_NAME)
print("Maximum sequence length:", MAX_LENGTH)
print("Number of classes:", NUM_CLASSES)

Model: bert-base-uncased
Maximum sequence length: 512
Number of classes: 5


In [ ]:
#Initialize the token
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully.
Vocabulary size: 30522


In [ ]:
#Creating DataSet Class
class MedicalAbstractDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=512):

        self.texts = dataframe["medical_abstract"].tolist()

        self.labels = (
            dataframe["condition_label"].values - 1
        ).astype(np.int64)

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        text = self.texts[index]
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
#Create DataSet Object
train_dataset = MedicalAbstractDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

val_dataset = MedicalAbstractDataset(
    val_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = MedicalAbstractDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 9817
Validation dataset: 1733
Test dataset: 2888


In [ ]:
#Create DataLoaders
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 1228
Validation batches: 217
Test batches: 361


In [ ]:
#Verifying One Batch
batch = next(iter(train_loader))

print("Input IDs shape     :", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape        :", batch["labels"].shape)

print("\nInput IDs dtype     :", batch["input_ids"].dtype)
print("Attention mask dtype:", batch["attention_mask"].dtype)
print("Labels dtype        :", batch["labels"].dtype)

print("\nLabels:", batch["labels"])

Input IDs shape     : torch.Size([8, 512])
Attention mask shape: torch.Size([8, 512])
Labels shape        : torch.Size([8])

Input IDs dtype     : torch.int64
Attention mask dtype: torch.int64
Labels dtype        : torch.int64

Labels: tensor([4, 1, 3, 0, 4, 2, 4, 4])


In [ ]:
#Defining Bert-Base Classification Model
class MedicalBERTClassifier(nn.Module):

    def __init__(
        self,
        model_name="bert-base-uncased",
        num_classes=5
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_classes
        )

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(cls_embedding)

        return logits

In [ ]:
#Initialize Model
model_scheduler = MedicalBERTClassifier(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES
).to(device)

print("Fresh BERT model initialized.")
print("Model device:", next(model_scheduler.parameters()).device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fresh BERT model initialized.
Model device: cuda:0


In [ ]:
#Defining Loss Function
criterion_scheduler = nn.CrossEntropyLoss()

print("Loss function:", criterion_scheduler)

Loss function: CrossEntropyLoss()


In [ ]:
#Define AdamW Optimizer
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

optimizer_scheduler = AdamW(
    model_scheduler.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Learning rate: 2e-05
Weight decay: 0.01


In [ ]:
#Configure Schedule and Warm-up
NUM_EPOCHS = 8

steps_per_epoch = len(train_loader)

total_training_steps = (
    steps_per_epoch * NUM_EPOCHS
)

warmup_steps = int(
    0.10 * total_training_steps
)

scheduler = get_linear_schedule_with_warmup(
    optimizer_scheduler,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Steps per epoch   :", steps_per_epoch)
print("Total steps       :", total_training_steps)
print("Warm-up steps     :", warmup_steps)

Steps per epoch   : 1228
Total steps       : 9824
Warm-up steps     : 982


In [ ]:
#Define Training Function
def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    scheduler,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, labels)

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # Update learning rate
        scheduler.step()

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            (predictions == labels)
            .sum()
            .item()
        )

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
#Define Validation Function
def evaluate_model(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True
            )

            attention_mask = batch["attention_mask"].to(
                device,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                non_blocking=True
            )

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item() * labels.size(0)
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    total = len(all_labels)

    epoch_loss = running_loss / total

    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return (
        epoch_loss,
        epoch_accuracy,
        epoch_macro_f1
    )

In [ ]:
#Training Loop
PATIENCE = 2

best_val_macro_f1 = -float("inf")
patience_counter = 0

for epoch in range(NUM_EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model_scheduler,
        train_loader,
        criterion_scheduler,
        optimizer_scheduler,
        scheduler,
        device
    )

    val_loss, val_accuracy, val_macro_f1 = evaluate_model(
        model_scheduler,
        val_loader,
        criterion_scheduler,
        device
    )

    current_lr = optimizer_scheduler.param_groups[0]["lr"]

    print(f"\nEpoch [{epoch + 1}/{NUM_EPOCHS}]")
    print("-" * 60)

    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy * 100:.2f}%")

    print(f"Val Loss       : {val_loss:.4f}")
    print(f"Val Accuracy   : {val_accuracy * 100:.2f}%")
    print(f"Val Macro-F1   : {val_macro_f1:.4f}")

    print(f"Learning Rate  : {current_lr:.8f}")

    # Save best model
    if val_macro_f1 > best_val_macro_f1:

        best_val_macro_f1 = val_macro_f1
        patience_counter = 0

        torch.save(
            model_scheduler.state_dict(),
            "best_medical_bert_scheduler.pt"
        )

        print("✓ Best scheduler model saved.")

    else:

        patience_counter += 1

        print(
            f"No improvement. "
            f"Patience: {patience_counter}/{PATIENCE}"
        )

        if patience_counter >= PATIENCE:

            print("\nEarly stopping triggered.")
            break


Epoch [1/8]
------------------------------------------------------------
Train Loss     : 1.1152
Train Accuracy : 53.26%
Val Loss       : 0.8911
Val Accuracy   : 63.47%
Val Macro-F1   : 0.6256
Learning Rate  : 0.00001944
✓ Best scheduler model saved.

Epoch [2/8]
------------------------------------------------------------
Train Loss     : 0.8453
Train Accuracy : 64.04%
Val Loss       : 0.8442
Val Accuracy   : 63.94%
Val Macro-F1   : 0.6347
Learning Rate  : 0.00001667
✓ Best scheduler model saved.

Epoch [3/8]
------------------------------------------------------------
Train Loss     : 0.7227
Train Accuracy : 68.92%
Val Loss       : 0.9055
Val Accuracy   : 61.80%
Val Macro-F1   : 0.5873
Learning Rate  : 0.00001389
No improvement. Patience: 1/2

Epoch [4/8]
------------------------------------------------------------
Train Loss     : 0.6303
Train Accuracy : 71.96%
Val Loss       : 0.9907
Val Accuracy   : 59.78%
Val Macro-F1   : 0.5897
Learning Rate  : 0.00001111
No improvement. Patien